In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader
from terramesh import build_terramesh_dataset
from plotting_utils import s2_to_rgb

from utils import build_model_from_registry, generate_outputs

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"{device = }")

In [ ]:
input_modality = 'LULC'
output_modality = 'S2L2A'

In [ ]:
base_url = "https://huggingface.co/datasets/ibm-esa-geospatial/TerraMesh/resolve/main/val"

urls = "::".join([
    f"{base_url}/[{input_modality},{output_modality}]/majortom_shard_{{000001..000008}}.tar",
    f"{base_url}/[{input_modality},{output_modality}]/ssl4eos12_shard_000009.tar"
])

In [ ]:
model = build_model_from_registry(
    input_modalities=[input_modality],
    output_modalities=[output_modality],
)

model = model.to(device)

In [ ]:
dataset = build_terramesh_dataset(
    path='',
    modalities=[input_modality, output_modality],
    split='val',
    urls=urls,
    shuffle=False,
    batch_size=8,
)

In [ ]:
dataloader = DataLoader(
    dataset, 
    batch_size=None, 
)

In [ ]:
n_samples = 5

data = {
    "input" : [],
    "original" : [],
    "generated" : [],
}

for i, batch in enumerate(dataloader):
    if i == n_samples: break

    _input = batch[input_modality].float()
    _input = torch.nn.functional.interpolate(_input, size=(224, 224), mode="nearest").to(device)
    _original = batch[output_modality].float()
    _original = torch.nn.functional.interpolate(_original, size=(224, 224), mode="nearest").to(device)

    data["input"].append(_input)
    data["original"].append(_original)


    

In [ ]:
generated = generate_outputs(model, data["input"], device=device)

In [ ]:
data["generated"] = list(map(lambda x: x[output_modality], generated))

In [ ]:


data["original"] = torch.from_numpy(np.array(data["original"], dtype="float32"))
data["original"] = [torch.nn.functional.interpolate(x, size=(224, 224)).to(device) for x in data["original"]]

In [ ]:
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance
from torchmetrics.image.inception import InceptionScore
from torchmetrics.image.ssim import StructuralSimilarityIndexMeasure
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity

ssim = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)
lpips = LearnedPerceptualImagePatchSimilarity(normalize=True).to(device)
inception_score = InceptionScore(normalize=True).to(device)
kid = KernelInceptionDistance(subset_size=n_samples, normalize=True).to(device)
fid = FrechetInceptionDistance(normalize=True).to(device)

In [ ]:
def normalize(x: np.ndarray):
    return x.astype(np.float32) / 255

In [ ]:
def evaluate(preds, targets):
    ssim_vals = []
    lpips_vals = []
    
    with torch.no_grad():
        for pred_batch, target_batch in zip(preds, targets):
           
            pred_rgb = np.array([s2_to_rgb(pred) for pred in pred_batch])
            target_rgb = np.array([s2_to_rgb(target) for target in target_batch])
            
            pred_rgb = torch.from_numpy(normalize(pred_rgb)).permute(0, 3, 1, 2).to(device)
            target_rgb = torch.from_numpy(normalize(target_rgb)).permute(0, 3, 1, 2).to(device)
            
            ssim_val = ssim(pred_rgb, target_rgb)
            ssim_vals.append(ssim_val)
            
            lpips_val = lpips(pred_rgb, target_rgb)
            lpips_vals.append(lpips_val)
            
            inception_score.update(pred_rgb)
            
            kid.update(target_rgb, real=True)
            kid.update(pred_rgb, real=False)
            
            fid.update(target_rgb, real=True)
            fid.update(pred_rgb, real=False)
            
            # torch.cuda.empty_cache()
        
        ssim_final = torch.stack(ssim_vals).mean().item()
        lpips_final = torch.stack(lpips_vals).mean().item()
        
        is_mean, is_std = inception_score.compute()
        is_mean, is_std = is_mean.item(), is_std.item()
        
        kid_mean, kid_std = kid.compute()
        kid_mean, kid_std = kid_mean.item(), kid_std.item()
        
        fid_val = fid.compute().item()
        
        inception_score.reset()
        kid.reset()
        fid.reset()
        
        results = {
            "SSIM": ssim_final,
            "LPIPS": lpips_final,
            "Inception Score (mean)": is_mean,
            "Inception Score (std)": is_std,
            "KID (mean)": kid_mean,
            "KID (std)": kid_std,
            "FID": fid_val,
        }
        return results

In [ ]:
result = evaluate(data["generated"], data["original"])

In [ ]:
result